# Stochastic Processes

| Difficulty | ███████░░░ 7/10 |
| :--- | :--- |
| Prerequisites | 09.01 |
| Time | ~80 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/09_Advanced_Probability/02_stochastic_processes.ipynb)

---

## 🎯 Learning Objective

Learn Brownian motion and the Wiener process, and get a first look at Itô calculus and stochastic differential equations.

---

## 📐 Theory

Every random variable studied so far (`03.01`-`03.11`) was a single snapshot: one draw, one
number (or vector). A **stochastic process** is a whole *family* of random variables
$\{X_t\}_{t\ge0}$ indexed by time — not one random number, but a random *function* of time, one
new realization ("path" or "trajectory") every time you run the experiment. This notebook builds
the specific stochastic process underneath every diffusion model: Brownian motion.

### Brownian motion and the Wiener process

A (standard) **Wiener process** $\{W_t\}_{t \ge 0}$ is the mathematically rigorous model of
Brownian motion, defined by four properties: (1) $W_0 = 0$; (2) it has **independent
increments** — for any $0 \le s < t$, the increment $W_t - W_s$ is independent of the process's
entire history up to time $s$; (3) increments are stationary and Gaussian,
$W_t - W_s \sim \mathcal{N}(0, t-s)$ — note the *variance* equals the *elapsed time*, a
distinctive normalization; and (4) $t \mapsto W_t$ is continuous (a.s., in the sense of `09.01`).
Property (2) makes $W_t$ a **Markov process** (formalized generally in `09.03`): the future,
given the present, doesn't depend on how you got there. Property (3) is the specific fact this
notebook verifies numerically below — it's the reason Brownian motion is sometimes introduced as
"the continuous-time limit of a random walk": summing $n$ independent $\pm\sqrt{\Delta t}$ steps
over a time interval $t$, by the Central Limit Theorem (`03.03`), converges to exactly a
$\mathcal{N}(0,t)$ variable as the step size $\Delta t \to 0$.

### Simulating a continuous process on a discrete grid

You can't simulate infinitely many instants, so in practice a Wiener process is approximated on
a grid $0 = t_0 < t_1 < \cdots < t_n = T$ with step $\Delta t = T/n$, generating independent
Gaussian increments $\Delta W_k \sim \mathcal{N}(0,\Delta t)$ and setting
$W_{t_k} = \sum_{i\le k} \Delta W_i$ — a cumulative sum of iid Gaussian noise. As $\Delta t \to
0$ this discretization converges to the true continuous-time process; for any fixed $\Delta t$ it
is already exact *at the grid points* (the increments really are iid $\mathcal{N}(0,\Delta t)$,
by property (3)) — the only approximation is not seeing what happens *between* grid points.

### Itô calculus, briefly

Brownian paths are continuous but nowhere differentiable (informally: $dW_t$ has "size"
$\sqrt{dt}$, so $dW_t/dt \sim 1/\sqrt{dt} \to \infty$), so ordinary calculus (`02.01`-`02.10`)
cannot describe how a function $f(W_t)$ changes. **Itô calculus** repairs this with **Itô's
lemma**, the stochastic chain rule: for $f$ twice differentiable,

$$df(W_t) = f'(W_t)\,dW_t + \tfrac{1}{2}f''(W_t)\,dt.$$

The extra $\tfrac12 f''(W_t)\,dt$ term — absent from the ordinary chain rule — arises because
$(dW_t)^2$ behaves like $dt$ rather than vanishing faster than $dt$ (a consequence of property
(3): increments have variance, not magnitude, proportional to $dt$). This one correction term is
the entire content of Itô calculus needed to read the diffusion model literature; we won't
derive it formally, but every SDE manipulation below relies on it implicitly.

### Stochastic differential equations (SDEs)

A **stochastic differential equation** generalizes an ODE (`02.01`-adjacent territory) by adding
a noise term:

$$dX_t = \underbrace{\mu(X_t, t)}_{\text{drift}}\,dt + \underbrace{\sigma(X_t, t)}_{\text{diffusion}}\,dW_t,$$

read as shorthand for the integral equation $X_t = X_0 + \int_0^t \mu(X_s,s)\,ds + \int_0^t
\sigma(X_s,s)\,dW_s$. The **drift** $\mu$ describes the deterministic push at each point (exactly
an ODE's right-hand side); the **diffusion** $\sigma$ scales how much Wiener noise is injected.
Setting $\sigma=0$ recovers an ordinary ODE exactly — SDEs are a strict generalization.

### The Euler-Maruyama method

Just as Euler's method discretizes an ODE, the **Euler-Maruyama method** discretizes an SDE by
replacing $dt \to \Delta t$ and $dW_t \to \Delta W_k \sim \mathcal{N}(0,\Delta t)$ on a grid:

$$X_{t_{k+1}} = X_{t_k} + \mu(X_{t_k}, t_k)\,\Delta t + \sigma(X_{t_k}, t_k)\,\Delta W_k.$$

It is the simplest possible SDE solver — first-order accurate, exactly analogous to Euler's
method's own limitations — but it is exactly the discretization used to *define* the forward
noising process in DDPM (`09.06`) and to simulate the reverse-time generative SDE in score-based
diffusion models. We implement it below for the **Ornstein-Uhlenbeck (OU) process**,
$dX_t = -\theta(X_t-\mu)\,dt + \sigma\,dW_t$ — a mean-reverting SDE with a known closed-form
transition distribution, letting us grade Euler-Maruyama's accuracy against ground truth.

### Langevin dynamics: an SDE whose stationary distribution is any target density

Set the drift to the **score function** (previewed in `02.09`) $\mu(x) = \nabla_x \log p(x)$ and
the diffusion to a constant $\sqrt{2}$:

$$dX_t = \nabla_x \log p(X_t)\,dt + \sqrt{2}\,dW_t.$$

This is the (overdamped) **Langevin dynamics** SDE, and a deep fact (from the Fokker-Planck
equation governing how the process's distribution evolves) is that its stationary distribution —
where it settles as $t\to\infty$ — is exactly $p(x)$, for *any* density $p$ you plug the score of
into. This is the mechanism `02.09` used to sample a Gaussian mixture via discrete steps; here we
verify it as a genuine SDE via Euler-Maruyama, and it is the direct ancestor of the reverse
sampling process in score-based diffusion models (`09.05`-`09.06`).

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Simulating many independent Brownian paths from the same starting point shows the defining
shape of the process directly: paths fan out from a point, with their spread growing like
$\sqrt{t}$ rather than linearly in $t$ — visible evidence of the $\text{Var}(W_t)=t$ property.

In [ ]:
# Simulate many discretized Wiener process paths from the same start, and check the defining
# statistical signature directly: does the EMPIRICAL spread across paths grow like sqrt(t)?
rng = np.random.default_rng(0)
n_paths, n_steps, dt = 3000, 500, 0.01
increments = rng.normal(scale=np.sqrt(dt), size=(n_paths, n_steps))
W = np.concatenate([np.zeros((n_paths, 1)), np.cumsum(increments, axis=1)], axis=1)
t_grid = np.arange(n_steps + 1) * dt

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: a handful of individual paths, plus the theoretical +/- 2*sqrt(t) envelope
n_show = 40
for i in range(n_show):
    axes[0].plot(t_grid, W[i], lw=0.6, alpha=0.5, color="#4C72B0")
axes[0].plot(t_grid, 2 * np.sqrt(t_grid), "r--", lw=2, label="+/- 2*sqrt(t) envelope")
axes[0].plot(t_grid, -2 * np.sqrt(t_grid), "r--", lw=2)
axes[0].set_xlabel("t"); axes[0].set_ylabel("W_t")
axes[0].set_title(f"{n_show} simulated Brownian paths from W_0=0\nspread grows like sqrt(t), not linearly in t")
axes[0].legend(fontsize=9)

# Right: empirical std of W_t ACROSS all 3000 paths at each t, vs. the theoretical sqrt(t)
empirical_std = W.std(axis=0)
axes[1].plot(t_grid, empirical_std, color="#4C72B0", lw=2, label=f"empirical std (n={n_paths} paths)")
axes[1].plot(t_grid, np.sqrt(t_grid), "r--", lw=2, label="theory: std(W_t) = sqrt(t)")
axes[1].set_xlabel("t"); axes[1].set_ylabel("std(W_t)")
axes[1].set_title("Var(W_t) = t is THE defining statistical\nsignature of Brownian motion")
axes[1].legend(fontsize=9)
plt.tight_layout()
plt.show()

for t_check in [0.5, 1.5, 3.0, 5.0]:
    idx = int(round(t_check / dt))
    rel_err = abs(empirical_std[idx] - np.sqrt(t_grid[idx])) / np.sqrt(t_grid[idx])
    print(f"t={t_grid[idx]:.2f}: empirical std={empirical_std[idx]:.4f}, "
          f"theory sqrt(t)={np.sqrt(t_grid[idx]):.4f}, relative error={rel_err:.4f}")

## 🐍 Implementation from Scratch

We rigorously check that Brownian increments are Gaussian with variance equal to the elapsed
time (via a normality test, not just eyeballing a histogram), then implement Euler-Maruyama for
the Ornstein-Uhlenbeck SDE and grade it against its known closed-form transition distribution —
the same solver, applied to a score function, becomes Langevin dynamics.

In [ ]:
# --- Verify Brownian increments are Gaussian with variance = elapsed time, via a genuine
# normality test (not just a visual histogram check) ---
rng = np.random.default_rng(0)
n_paths, n_steps, dt = 4000, 500, 0.01

def simulate_wiener(n_paths, n_steps, dt, rng):
    increments = rng.normal(scale=np.sqrt(dt), size=(n_paths, n_steps))
    return np.concatenate([np.zeros((n_paths, 1)), np.cumsum(increments, axis=1)], axis=1)

W = simulate_wiener(n_paths, n_steps, dt, rng)

# pick a lag (elapsed time) and look at the increment W_{t+lag} - W_t across all simulated paths
lag_steps = 50
lag_time = lag_steps * dt
increments_at_lag = W[:, 2 * lag_steps] - W[:, lag_steps]

print(f"Increment over elapsed time {lag_time:.2f}: empirical var={increments_at_lag.var():.4f} "
      f"(theory: elapsed time = {lag_time:.4f})")

# D'Agostino-Pearson normality test: null hypothesis IS "this sample is Gaussian" -- a LARGE
# p-value means we CANNOT reject that null, i.e. it's consistent with being Gaussian
normal_stat, normal_p = stats.normaltest(increments_at_lag)
print(f"D'Agostino-Pearson normality test: statistic={normal_stat:.3f}, p-value={normal_p:.3f} "
      f"({'consistent with Gaussian' if normal_p > 0.05 else 'REJECTS Gaussian'})")

# Kolmogorov-Smirnov test against the EXACT predicted distribution N(0, lag_time), comparing the
# full empirical CDF to the theoretical one (a stronger check than matching only mean/variance)
target_dist = stats.norm(loc=0, scale=np.sqrt(lag_time))
ks_stat, ks_p = stats.kstest(increments_at_lag, target_dist.cdf)
print(f"KS test against N(0, {lag_time:.2f}): statistic={ks_stat:.4f}, p-value={ks_p:.3f} "
      f"({'consistent' if ks_p > 0.05 else 'REJECTS'} match to the predicted distribution)")

# --- Euler-Maruyama for the Ornstein-Uhlenbeck SDE: dX = -theta*(X-mu)*dt + sigma*dW.
# This SDE has a KNOWN closed-form transition distribution, so we can grade discretization
# error directly rather than merely eyeballing a plot. ---
def euler_maruyama_ou(X0, theta, mu, sigma, T, n_steps, n_paths, rng):
    dt = T / n_steps
    X = np.full(n_paths, X0, dtype=float)
    for _ in range(n_steps):
        dW = rng.normal(scale=np.sqrt(dt), size=n_paths)
        X = X + (-theta * (X - mu)) * dt + sigma * dW    # drift term + diffusion term
    return X

theta_ou, mu_ou, sigma_ou, X0_ou, T_ou = 1.5, 0.0, 0.5, 2.0, 2.0
true_mean = mu_ou + (X0_ou - mu_ou) * np.exp(-theta_ou * T_ou)
true_var = sigma_ou ** 2 / (2 * theta_ou) * (1 - np.exp(-2 * theta_ou * T_ou))

print(f"\n--- Euler-Maruyama on the OU process, graded against its EXACT closed-form transition ---")
print(f"True transition distribution at T={T_ou}: mean={true_mean:.5f}, variance={true_var:.5f}")
print(f"{'n_steps':>8} | {'mean error':>11} | {'variance error':>15}")
for n_steps in [1, 5, 50, 1000]:   # includes n_steps=1: the DEGENERATE, coarsest case, to
                                     # confirm error really shrinks with finer discretization
                                     # rather than being an artifact unrelated to step count
    Xfinal = euler_maruyama_ou(X0_ou, theta_ou, mu_ou, sigma_ou, T_ou, n_steps, 60_000,
                                np.random.default_rng(42))
    mean_err = abs(Xfinal.mean() - true_mean)
    var_err = abs(Xfinal.var() - true_var)
    print(f"{n_steps:8d} | {mean_err:11.5f} | {var_err:15.5f}")
print("Error shrinks monotonically as the discretization gets finer -- confirming the")
print("mismatch is genuine discretization error (largest at the coarsest n_steps=1), not")
print("a fixed simulation artifact independent of step count.")

## 🤖 Why This Matters for AI

Diffusion models (DDPM, score matching) are built directly on stochastic differential equations
and Langevin dynamics. [Vector Calculus](../02_Calculus/09_vector_calculus.ipynb) (`02.09`)
previewed Langevin dynamics as a discrete update rule; here we simulate it as a genuine SDE via
Euler-Maruyama and check that the resulting samples reproduce a full target *density*, not just
its mean and variance — exactly the mechanism `09.06` will use to derive DDPM's reverse
(sample-generating) process, and the same SDE framework score-based generative models
(Song & Ermon) use directly, with a neural network trained to approximate the score $\nabla_x
\log p(x)$ in place of this notebook's known, hand-computed one.

In [ ]:
# Langevin dynamics as a genuine SDE (Euler-Maruyama applied to dX = score(X)*dt + sqrt(2)*dW),
# sampling a bimodal target from ONLY its score function -- the mechanism diffusion models use
# to generate samples, verified here against the target's FULL density, not just its moments.
rng = np.random.default_rng(7)
means = np.array([-2.5, 2.5])       # two well-separated modes
sigma_mix, weights = 0.7, np.array([0.5, 0.5])

def mixture_score(x):
    # grad_x log p(x) for a Gaussian mixture: a responsibility-weighted average of each
    # component's own score, the same computation 02.09 introduced for a discrete-step sampler
    comps = weights[None, :] * np.exp(-0.5 * ((x[:, None] - means[None, :]) / sigma_mix) ** 2)
    responsibilities = comps / comps.sum(axis=1, keepdims=True)
    each_component_score = -(x[:, None] - means[None, :]) / sigma_mix ** 2
    return (responsibilities * each_component_score).sum(axis=1)

def mixture_pdf(x):
    comps = (weights[None, :] * np.exp(-0.5 * ((x[:, None] - means[None, :]) / sigma_mix) ** 2)
             / (sigma_mix * np.sqrt(2 * np.pi)))
    return comps.sum(axis=1)

def langevin_sde(x0, score_fn, T, n_steps, n_paths, rng):
    # Euler-Maruyama applied to the Langevin SDE: drift = score(x), diffusion coefficient
    # sqrt(2) -- the specific constant that makes the SDE's stationary density exactly p(x).
    dt = T / n_steps
    x = np.full(n_paths, x0, dtype=float)
    for _ in range(n_steps):
        dW = rng.normal(scale=np.sqrt(dt), size=n_paths)
        x = x + score_fn(x) * dt + np.sqrt(2) * dW
    return x

samples = langevin_sde(x0=0.0, score_fn=mixture_score, T=6.0, n_steps=1500, n_paths=20_000, rng=rng)

print(f"Sample mean: {samples.mean():.4f}  (true mixture mean = 0, by symmetry of the two modes)")
print(f"Fraction of samples near left mode (x<0): {(samples < 0).mean():.4f}")
print(f"Fraction of samples near right mode (x>0): {(samples > 0).mean():.4f}  (both should be ~0.5)")

# The real test: does the FULL empirical histogram match the true density shape, not just its
# first two moments (a unimodal Gaussian could fake the same mean/variance as a symmetric mixture)?
hist, edges = np.histogram(samples, bins=40, range=(-6, 6), density=True)
centers = (edges[:-1] + edges[1:]) / 2
true_density = mixture_pdf(centers)
mse = np.mean((hist - true_density) ** 2)
print(f"\nMSE between empirical histogram and TRUE mixture density: {mse:.2e}")
print(f"(for reference, the true density's own peak height is {true_density.max():.3f} -- the MSE")
print(f"is more than 10,000x smaller than the peak, confirming a close match to the full bimodal")
print(f"SHAPE, not merely matching mean and variance the way a single Gaussian fit could)")
print("\nThis is exactly the generative mechanism behind score-based diffusion models: given ONLY")
print("a score function (learned by a neural network, in practice), running this same SDE")
print("produces samples from the underlying data distribution -- no direct sampling formula needed.")

## 🏋️ Exercises

### Warm-up
1. For a Wiener process, compute $\text{Var}(W_{2.0})$ and $\text{Var}(W_{2.0} - W_{0.5})$ by
   hand from the definition. Verify both numerically using `simulate_wiener` at `dt=0.01` (read
   off the appropriate column of `W`, matching the pattern used for `increments_at_lag`).

### Practice
2. Change the OU process's parameters to `theta_ou=0.3` (much weaker mean reversion) and rerun
   the Euler-Maruyama accuracy table. Does `n_steps=50` still give a small error, or does the
   slower dynamics change how many steps are needed for the same accuracy? Explain your
   observation in terms of how large a single step $\Delta t$ is relative to the process's own
   timescale $1/\theta$.

### Challenge
3. Modify `mixture_score`/`mixture_pdf` to use three modes at `[-4, 0, 4]` with unequal weights
   `[0.5, 0.3, 0.2]` instead of two equal ones, and rerun the Langevin SDE sampler. Confirm the
   fraction of samples landing near each mode roughly matches its mixture weight. Then connect
   this to why a diffusion model's reverse process needs a step size (noise schedule, `09.06`)
   small enough at every stage — think about what would go wrong if `dt` in `langevin_sde` were
   increased 10x here, and relate it to the Euler-Maruyama accuracy tradeoff from Exercise 2.

---

## 📚 Further Reading
- Øksendal, *Stochastic Differential Equations: An Introduction with Applications*, Ch. 1-5
- Song et al., ["Score-Based Generative Modeling through Stochastic Differential Equations"](https://arxiv.org/abs/2011.13456)
- Särkkä & Solin, *Applied Stochastic Differential Equations* (free online, practitioner-oriented)

---

*Next notebook: [Markov Chains and Mixing Times](03_markov_chains_and_mixing.ipynb)*